# FIT5202 : Recommendation using Collaborative Filtering

Collaborative filtering (CF) is a technique commonly used to build personalized recommendations on the Web. Some popular websites that make use of the collaborative filtering technology include Amazon, Netflix, iTunes, IMDB, LastFM, Delicious and StumbleUpon. In collaborative filtering, algorithms are used to make automatic predictions about a user's interests by compiling preferences from several users.

In this lab, our task is to use a collaborative algorithm to recommend top artists from the given dataset. The dataset can be downloaded from Moodle. 
<p style="color:red">Complete the required tasks in the tutorial. The activites are denoted as "Task" with the required instructions.</p>
<br/>

## Table of Contents

* [ALS Lecture Demo](#als-demo)
* [Use-Case Music Recommendation](#use-case)
    * [Data Loading](#data-loading)
    * [Data Preparation](#data-prep)
    * [Data Exploration](#data-exploration)
    * [Train-Test Split](#train-test-split)
    * [Model Building](#model-building)
    * [Evaluation](#evaluation)
    * [Hyperparameter Tuning and Cross Validation](#cv)
    * [Making Predictions](#predictions)    
* [Lab Tasks](#lab-task-1)
    * [Lab Task 1](#lab-task-1)
    * [Lab Task 2](#lab-task-2)
    * [Lab Task 3](#lab-task-3)
    * [Lab Task 4](#lab-task-4)
    * [Lab Task 5](#lab-task-5)
    * [Lab Task 6](#lab-task-6)
    
## Including Libraries and Initializing Spark Context

In [1]:
#import libraries
from pyspark import SparkContext
from pyspark.ml.recommendation import ALS
from pyspark.sql import SparkSession ,Row
from pyspark.sql.functions import col,split
from pyspark.sql.types import StructType,StructField,IntegerType,StringType


appName="Collaborative Filtering with PySpark"
#initialize the spark session
spark = SparkSession.builder.appName(appName).getOrCreate()
#get sparkcontext from the sparksession
sc = spark.sparkContext

# Alternating Least Squares DEMO

Please go through the ALS Demo presented in the Lecture to understand the basic flow before starting with the lab tasks.

## Case Study : Music Recommendation
The goal here is to use the data provided to create a recommendation system using collaborative filtering using the social influence data and predict artists a user might like but have not listened to.

> Consider the following example. If user A is a neighbor of user B, and they have similar musical tastes, then there is a very strong tie between them. If user B is a big fan of artist C, and has scrobbled them numerous times, then there is also a strong tie between them. Based on last.fm’s data, user A has not yet listened to artist C (no link has formed between them yet), and there is a good chance that user A will also like artist C.<a href="https://blogs.cornell.edu/info2040/2012/09/20/last-fm-music-reccomendation-incorporating-social-network-ties-and-collaborative-filtering/#:~:text=their%20listening%20frequency.-,Last.,in%20the%20user's%20local%20network." target="_BLANK">Ref</a>


The original dataset is available at <a href="https://www.last.fm/api/" target="_blank">last.fm api</a>. The dataset provided here is a lighter version, resized for the sake of simplicity. The dataset contains three files as follows:
<ul>
    <li><strong>user_artist_data.txt</strong>
        3 columns: <code>user_id, artist_id, playcount</code></li>
    <li><strong>artist_data.txt</strong>
        2 columns: <code>artist_id ,artist_name</code></li>
    <li><strong>artist_alias.txt</strong>
        2 columns: <code>bad_id, good_id</code>
        [known incorrectly spelt artists and the correct artist id].</li>
</ul>


<a class="anchor" id="lab-task-1"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">1. Lab Task: </strong> 
Import the two other files (user_artist_data.txt and artist_alias.txt) to create two dataframes <code>df_artist_alias</code> and <code>df_user_artist</code>
    
<strong style="color:red">NOTE:</strong> Check the <strong>delimiter</strong> used in these files. <code>\t</code> may not be used for all files.
</div>.




### Data Loading 

In [31]:
df = spark.read.text("user_artist_data.txt")
split_col = split(df['value'], ' ')
df = df.withColumn('user_id', split_col.getItem(0))
df = df.withColumn('artist_id', split_col.getItem(1))
df = df.withColumn('playcount', split_col.getItem(2))
df_user_artist=df.drop('value')
df_user_artist.show()

+-------+---------+---------+
|user_id|artist_id|playcount|
+-------+---------+---------+
|1059637|  1000010|      238|
|1059637|  1000049|        1|
|1059637|  1000056|        1|
|1059637|  1000062|       11|
|1059637|  1000094|        1|
|1059637|  1000112|      423|
|1059637|  1000113|        5|
|1059637|  1000114|        2|
|1059637|  1000123|        2|
|1059637|  1000130|    19129|
|1059637|  1000139|        4|
|1059637|  1000241|      188|
|1059637|  1000263|      180|
|1059637|  1000289|        2|
|1059637|  1000305|        1|
|1059637|  1000320|       21|
|1059637|  1000340|        1|
|1059637|  1000427|       20|
|1059637|  1000428|       12|
|1059637|  1000433|       10|
+-------+---------+---------+
only showing top 20 rows



In [3]:
df = spark.read.text("artist_data.txt")
split_col = split(df['value'], '\t')
df = df.withColumn('artist_id', split_col.getItem(0))
df = df.withColumn('artist_name', split_col.getItem(1))
df_artist=df.drop('value')
# df_artist.show()

In [4]:
df = spark.read.text("artist_alias.txt")
split_col = split(df['value'], '\t')
df = df.withColumn('artist_id', split_col.getItem(0))
df = df.withColumn('artist_name', split_col.getItem(1))
df_artist_alias=df.drop('value')

In [5]:
df_artist_alias.show()

+---------+-----------+
|artist_id|artist_name|
+---------+-----------+
|  1027859|    1252408|
|  1017615|        668|
|  6745885|    1268522|
|  1018110|    1018110|
|  1014609|    1014609|
|  6713071|       2976|
|  1014175|    1014175|
|  1008798|    1008798|
|  1013851|    1013851|
|  6696814|    1030672|
|  1036747|    1239516|
|  1278781|    1021980|
|  2035175|    1007565|
|  1327067|    1308328|
|  2006482|    1140837|
|  1314530|    1237371|
|  1160800|    1345290|
|  1255401|    1055061|
|  1307351|    1055061|
|  1234249|    1005225|
+---------+-----------+
only showing top 20 rows



### Data Preparation 

The <code>df_user_artist</code> contains <strong>bad ids</strong>, so the <strong>bad ids</strong> in the <code>user_artist_data.txt</code> file need to be remapped to <strong>goodids</strong>. The first task is to create a dictionary of the artist_alias, so that it can be passed over a broadcast variable.

Broadcast makes Spark send and hold in memory just one copy for each executor in the cluster. When there are thousands of tasks, and many execute in parallel on each executor, this can save significant network traffic and memory.
But you cannot directly broadcast a dataframe, it has to be converted to a list first

In [6]:
#After loading the artist_alias data to the dataframe, it is converted to a dictionary to be set as a broadcast variable
artist_alias = dict(df_artist_alias.collect())

<a class="anchor" id="lab-task-2"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">2. Lab Task: </strong> 
For the dictionary <strong>artistAlias</strong> which contains key value pair of badid and goodid, create a broadcast variable called <strong>bArtistAlias</strong>.
</div>

In [7]:
#Write the code below
bArtistAlias = sc.broadcast(artist_alias)

After the broadcast variable is created, a function to replace the badids by looking up the values from the broadcasted dictionary is implemented for the userArtistRDD.


In [8]:
from pyspark.sql.functions import udf, struct
def lookup_correct_id(artist_id):    
    finalArtistID = bArtistAlias.value.get(artist_id)
    if finalArtistID is None:
        finalArtistID = artist_id
    return finalArtistID    

lookup_udf = udf(lookup_correct_id, StringType())

<a class="anchor" id="lab-task-3"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">3. Lab Task: </strong> 
    Apply the udf <code>lookup_udf</code> on the column of 'artist_id' to replace the "badids" in the <code>df_user_artist</code> dataframe.
</div>

In [9]:
#Write your code below
df_user_artist=df_user_artist.withColumn("artist_id",lookup_udf("artist_id"))

When we want to repeteadly access a dataframe or an RDD, it is a good idea to cache them, it helps to speed up applications.

In [10]:
#Uncomment this to use caching
#df_user_artist.cache()

In [11]:
df_artist.show()

+---------+--------------------+
|artist_id|         artist_name|
+---------+--------------------+
|  1240105|        André Visior|
|  1240113|           riow arai|
|  1240132|Outkast & Rage Ag...|
|  6776115|            小松正夫|
|  1030848|      Raver's Nature|
|  6671601|      Erguner, Kudsi|
|  1106617|              Bloque|
|  1240185|      Lexy & K. Paul|
|  6671631|    Rev. W.M. Mosley|
|  6671632|      Labelle, Patti|
|  1240238|   the Chinese Stars|
|  1240262|            The Gufs|
|  6718605|          Bali Music|
|  6828988|Southern Conferen...|
|  1240415|        Paul & Paula|
|  1009439|            Cinnamon|
|  1018275|      School Of Fish|
|  6671680|Armstrong, Louis ...|
|  1240508|The Ozark Mountai...|
|  1240510| The Mercury Program|
+---------+--------------------+
only showing top 20 rows



In [12]:
#Check if any columns in the dataframes have null values in them, if they contain null values, drop the rows
df_artist.where(col("artist_id").isNull()).count()

0

## Data Exploration <a class="anchor" name="data-exploration"></a>

<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px">
    The function below returns top <strong>N</strong> artist for a user with user_id : <code>2062243</code> by joining <code>user</code> and <code>user_artist</code> datasets on the common key <code>artist_id</code>.
</div>



In [13]:
def top_n_artists(artist,user_artist,user_id,limit):
    '''Returns top n artists liked by a particular user'''
    '''Takes artist,user_artist, user_id and limit as input'''
    
    df = artist.join(user_artist,artist.artist_id==user_artist.artist_id)\
            .filter(user_artist.user_id==user_id)\
            .sort(user_artist.playcount.desc())\
            .select(user_artist.user_id,user_artist.playcount,artist.artist_name)\
            .limit(limit)
    return df  
top_n_artists(df_artist,df_user_artist,2062243,60).show(truncate=False)

+-------+---------+----------------------------+
|user_id|playcount|artist_name                 |
+-------+---------+----------------------------+
|2062243|99       |morgan heritage             |
|2062243|98       |Moxy Früvous                |
|2062243|98       |Music 205nders              |
|2062243|98       |Music 205tills, Nash & Young|
|2062243|98       |Music 205lub                |
|2062243|98       |Music 205n                  |
|2062243|98       |James Brown                 |
|2062243|98       |Nirvana                     |
|2062243|98       |Music 205olf                |
|2062243|98       |Mountain                    |
|2062243|98       |Mr C The Slide Man          |
|2062243|98       |Music 205its                |
|2062243|98       |Music 205Satins             |
|2062243|98       |Money Mark                  |
|2062243|98       |Moguai                      |
|2062243|98       |La Bouche                   |
|2062243|98       |Mortal Combat Soundtrack    |
|2062243|98       |M

Since the data in RDDs is string type, we need to convert them into numeric type for using it in the Recommendation Algorithm

In [14]:
#Cast the data column into integer types
for col_name in df_user_artist.columns:
    df_user_artist = df_user_artist.withColumn(col_name, df_user_artist[col_name].cast(IntegerType()))

df_artist = df_artist.withColumn('artist_id', df_artist['artist_id'].cast(IntegerType()))

In [15]:
df_user_artist.printSchema()
# df_user_artist.show()

root
 |-- user_id: integer (nullable = true)
 |-- artist_id: integer (nullable = true)
 |-- playcount: integer (nullable = true)



In [16]:
df_artist.printSchema()
# df_artist.show()

root
 |-- artist_id: integer (nullable = true)
 |-- artist_name: string (nullable = true)



<a class="anchor" id="lab-task-4"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">4. Lab Task: </strong> 
Create a deterministic 80/20 split of training and testing dataset.
</div>

### Train Test Split

In [17]:
#Write your code here
(train, test) = df_user_artist.randomSplit([0.8, 0.2], 2023)

## Model Building <a href="https://spark.apache.org/docs/latest/ml-collaborative-filtering.html" target="_blank">[REF]</a> <a class="anchor" name="model-building"></a>
Collaborative filtering is commonly used for recommender systems. These techniques aim to fill in the missing entries of a user-item association matrix. spark.ml currently supports model-based collaborative filtering, in which users and products are described by a small set of latent factors that can be used to predict missing entries. spark.ml uses the alternating least squares (ALS) algorithm to learn these latent factors. The implementation in spark.ml has the following parameters:

- <strong>numBlocks</strong> is the number of blocks the users and items will be partitioned into in order to parallelize computation (defaults to 10).
- <strong>rank</strong> is the number of latent factors in the model (defaults to 10).
- <strong>maxIter</strong> is the maximum number of iterations to run (defaults to 10).
- <strong>regParam</strong> specifies the regularization parameter in ALS (defaults to 1.0).
- <strong>implicitPrefs</strong> specifies whether to use the explicit feedback ALS variant or one adapted for implicit feedback data (defaults to false which means using explicit feedback).
- <strong>alpha</strong> is a parameter applicable to the implicit feedback variant of ALS that governs the baseline confidence in preference observations (defaults to 1.0).
- <strong>nonnegative</strong> specifies whether or not to use nonnegative constraints for least squares (defaults to false).

In [18]:
als = ALS(maxIter=5, implicitPrefs=True, alpha=40,userCol="user_id", itemCol="artist_id", ratingCol="playcount",
          coldStartStrategy="drop")

<a class="anchor" id="lab-task-5"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">5. Lab Task: </strong> 
Perform the following tasks.
    <ul><li>Train the model with the training set created from above.</li><li> Then transform use the test data to get the predictions. </li><li>Display the first 20 predictions from the results.</li></ul>    
<i>The predictions shown below will be just indicator of how closely a given artist will be to the user's existing preferences</i>
</div>

In [19]:
#Write your code below
model = als.fit(train)
# predictions.filter($"artist_id">0).show()
predictions = model.transform(test)
# predictions.filter(predictions.artist_id<0).show()

In [20]:
type(predictions)

pyspark.sql.dataframe.DataFrame

In [21]:
train.show()
predictions.show()

+-------+---------+---------+
|user_id|artist_id|playcount|
+-------+---------+---------+
|1000647|       18|        2|
|1000647|       28|      198|
|1000647|       40|       73|
|1000647|       59|       22|
|1000647|       61|       17|
|1000647|       76|      336|
|1000647|       83|       15|
|1000647|       91|        1|
|1000647|      189|       16|
|1000647|      202|       15|
|1000647|      207|        1|
|1000647|      224|       19|
|1000647|      244|       10|
|1000647|      251|       75|
|1000647|      276|       13|
|1000647|      313|      302|
|1000647|      341|        2|
|1000647|      358|        1|
|1000647|      393|        5|
|1000647|      405|       15|
+-------+---------+---------+
only showing top 20 rows

+-------+---------+---------+------------+
|user_id|artist_id|playcount|  prediction|
+-------+---------+---------+------------+
|1059637|      463|       55|   3.6442192|
|1024631|      833|        5|  0.30479744|
|1024631|     4935|        6|  0.279454

## Evalutation of ALS

We can evaluate ALS using RMSE (Root Mean Squared Error) using the RegressionEvaluator as shown below:

In [22]:
#Write your code here
from pyspark.ml.evaluation import RegressionEvaluator

evaluator = RegressionEvaluator(metricName="rmse", labelCol="playcount",
                                predictionCol="prediction")
rmse = evaluator.evaluate(predictions)
print("Root-mean-square error = " + str(rmse))

Root-mean-square error = 903.975485310347


<strong style="color:red">NOTE: </strong>If you run the above code, the RMSE you will observe is very high. 

For implicit data, RMSE is not a reliable score since, we don't have any reliable feedback over if items are disliked. RMSE requires knowing which items the user dislikes. Spark does not have a readily available solution for to evaluate the implicit data. The following function implements ROEM (Rank Ordering Error Metric) on the prediction data. You can refer to the details about this <a href="https://campus.datacamp.com/courses/recommendation-engines-in-pyspark/what-if-you-dont-have-customer-ratings?ex=6" target="_BLANK">here</a>.

In [23]:
# predictions.groupBy().sum(ratingCol).collect()[0][0]

In [24]:
def ROEM(predictions, userCol = "userId", itemCol = "songId", ratingCol = "num_plays"):
    #Creates table that can be queried
    predictions.createOrReplaceTempView("predictions")

    #Sum of total number of plays of all songs
    denominator = predictions.groupBy().sum(ratingCol).collect()[0][0]

    #Calculating rankings of songs predictions by user
    spark.sql("SELECT " + userCol + " , " + ratingCol + " , PERCENT_RANK() OVER (PARTITION BY " + userCol + " ORDER BY prediction DESC) AS rank FROM predictions").createOrReplaceTempView("rankings")

    #Multiplies the rank of each song by the number of plays and adds the products together
    numerator = spark.sql('SELECT SUM(' + ratingCol + ' * rank) FROM rankings').collect()[0][0]

    performance = numerator/denominator

    return performance

In [25]:
ROEM(predictions,'user_id','artist_id','playcount')

0.4556980200128001

### Hyperparameter tuning and cross validation

Since we can't use RMSE as the evaluation metric for "implicit data", we need to manually implement the hyperparameter tuning for the ALS. This code is adapted from the following source [<a href="https://github.com/jamenlong/ALS_expected_percent_rank_cv/blob/master/ROEM_cv.py" target="_BLANK">ref</a>]

<code>alpha</code> is an important hyper-parameter for ALS with implicit feedback. It governs the baseline confidence in preference observations. It is a way to assign a confidence values to the <code>playcount</code>. Higher <code>playcount</code> would mean that we have higher confidence that the user likes that artist and lower <code>playcount</code> would mean the user doesn't like that much.

In [26]:
def ROEM_cv(df, userCol = "user_id", itemCol = "artist_id", ratingCol = "playcount", ranks = [10], maxIters = [10], regParams = [.05], alphas = [10, 40]):
  
    from pyspark.sql.functions import rand
    from pyspark.ml.recommendation import ALS

    ratings_df = df.orderBy(rand()) #Shuffling to ensure randomness

    #Building train and validation test sets
    train, validate = df.randomSplit([0.8, 0.2], seed = 0)

    #Building 3 folds within the training set.
    test1, test2,test3 = train.randomSplit([0.33,0.33,0.33], seed = 1)
    train1 = test2.union(test3)
    train2 = test1.union(test2)
    train3 = test1.union(test3)
    

    #Creating variables that will be replaced by the best model's hyperparameters for subsequent printing
    best_validation_performance = 9999999999999
    best_rank = 0
    best_maxIter = 0
    best_regParam = 0
    best_alpha = 0
    best_model = 0
    best_predictions = 0

      #Looping through each combindation of hyperparameters to ensure all combinations are tested.
    for r in ranks:
        for mi in maxIters:
            for rp in regParams:
                for a in alphas:
                #Create ALS model
                    als = ALS(rank = r, maxIter = mi, regParam = rp, alpha = a, userCol=userCol, itemCol=itemCol, ratingCol=ratingCol,
                            coldStartStrategy="drop", nonnegative = True, implicitPrefs = True)

                    #Fit model to each fold in the training set
                    model1 = als.fit(train1)
                    model2 = als.fit(train2)
                    model3 = als.fit(train3)
                    
                    #Generating model's predictions for each fold in the test set
                    predictions1 = model1.transform(test1)
                    predictions2 = model2.transform(test2)
                    predictions3 = model3.transform(test3)
                    
                    #Expected percentile rank error metric function
                    def ROEM(predictions, userCol = userCol, itemCol = itemCol, ratingCol = ratingCol):
                        #Creates table that can be queried
                        predictions.createOrReplaceTempView("predictions")

                        #Sum of total number of plays of all songs
                        denominator = predictions.groupBy().sum(ratingCol).collect()[0][0]

                        #Calculating rankings of songs predictions by user
                        spark.sql("SELECT " + userCol + " , " + ratingCol + " , PERCENT_RANK() OVER (PARTITION BY " + userCol + " ORDER BY prediction DESC) AS rank FROM predictions").createOrReplaceTempView("rankings")

                        #Multiplies the rank of each song by the number of plays and adds the products together
                        numerator = spark.sql('SELECT SUM(' + ratingCol + ' * rank) FROM rankings').collect()[0][0]

                        performance = numerator/denominator

                        return performance

                    #Calculating expected percentile rank error metric for the model on each fold's prediction set
                    performance1 = ROEM(predictions1)
                    performance2 = ROEM(predictions2)
                    performance3 = ROEM(predictions3)
                    

                    #Printing the model's performance on each fold        
                    print("Model Parameters: \nRank:", r,"\nMaxIter:", mi, "\nRegParam:",rp,"\nAlpha: ",a)
                    print("Test Percent Rank Errors: ", performance1, performance2, performance3)

                    #Validating the model's performance on the validation set
                    validation_model = als.fit(train)
                    validation_predictions = validation_model.transform(validate)
                    validation_performance = ROEM(validation_predictions)

                    #Printing model's final expected percentile ranking error metric
                    print("Validation Percent Rank Error: "), validation_performance
                    print(" ")

                    #Filling in final hyperparameters with those of the best-performing model
                    if validation_performance < best_validation_performance:
                        best_validation_performance = validation_performance
                        best_rank = r
                        best_maxIter = mi
                        best_regParam = rp
                        best_alpha = a
                        best_model = validation_model
                        best_predictions = validation_predictions

    #Printing best model's expected percentile rank and hyperparameters
    print ("**Best Model** ")
    print ("  Percent Rank Error: ", best_validation_performance)
    print ("  Rank: ", best_rank)
    print ("  MaxIter: ", best_maxIter)
    print ("  RegParam: ", best_regParam)
    print ("  Alpha: ", best_alpha)
    
    return best_model, best_predictions

In [27]:
ROEM_cv(df_user_artist)

Model Parameters: 
Rank: 10 
MaxIter: 10 
RegParam: 0.05 
Alpha:  10
Test Percent Rank Errors:  0.2502066095686637 0.14947593743079976 0.18246693141546544
Validation Percent Rank Error: 
 
Model Parameters: 
Rank: 10 
MaxIter: 10 
RegParam: 0.05 
Alpha:  40
Test Percent Rank Errors:  0.25286360108130806 0.19509661193609804 0.20776432169052744
Validation Percent Rank Error: 
 
**Best Model** 
  Percent Rank Error:  0.34837235414554046
  Rank:  10
  MaxIter:  10
  RegParam:  0.05
  Alpha:  10


(ALSModel: uid=ALS_956e8e42b130, rank=10,
 DataFrame[user_id: int, artist_id: int, playcount: int, prediction: float])

## Making Predictions
The k-fold validation implement might take long time to run. You can use the initial ALS model to make the predictions.
Assuming you have successfully trained the model, we want to now use the model to <strong>find top Artists recommended for each user</strong>. We can use the <i><strong>recommendForAllUsers</strong></i> function available in the ALS model to get the list of top recommendations for each users. You can further explore the details of the API <a href="https://spark.apache.org/docs/2.2.0/api/python/pyspark.ml.html#pyspark.ml.recommendation.ALS" target="_blank">here</a>.

The <code>recommendForAllUsers</code> only gives the list of artist_ids for the users, you can write the code to map these artist_ids back to their names.

The function below takes userId and limit as the input. For the given userId, it gets the list of current top liked artists(based on the playcount).


<a class="anchor" id="lab-task-6"></a>
<div style="background:rgba(0,109,174,0.2);padding:10px;border-radius:4px"><strong style="color:#FF5555">6. Lab Task: </strong> 
    Write a function to find the top <strong>N</strong> recommended artists for the user : <strong>2062243</strong>. Display  <code>artist_id and artist_name</code> both.

In [28]:
# model.recommendForAllUsers(10).show(truncate=False)
model.recommendForAllUsers(10).toPandas()

,user_id,recommendations
0,1001440,"[(1009156, 5.575455665588379), (1004421, 4.987..."
1,1021940,"[(1009156, 11.317176818847656), (478, 6.520590..."
2,2005710,"[(1007735, 6.921533584594727), (1058104, 4.389..."
3,1058890,"[(1007735, 6.069391250610352), (606, 3.9017715..."
4,1017610,"[(1000199, 5.338196277618408), (1005386, 4.168..."
5,2007381,"[(388, 6.851072788238525), (1004294, 6.4003176..."
6,1073421,"[(1007735, 10.5175142288208), (756, 7.64436340..."
7,1024631,"[(1026440, 5.067617416381836), (1001909, 4.369..."
8,1021501,"[(2600, 12.188282012939453), (1007735, 7.84592..."
9,1035511,"[(1006633, 4.565447807312012), (1000199, 4.514..."


In [29]:
import pyspark.sql.functions as F
# A naive implementation, less efficient
def recommended_artist_naive(als_model,user_id,limit):
    seletecd_user = model.recommendForAllUsers(limit).filter(col('user_id')==user_id)

    recommendations = seletecd_user.select("recommendations").collect()

    topArtists = []
    for item in recommendations[0][0]:        
        topArtists.append(item.artist_id)
        
    schema = StructType([StructField("artist_id",IntegerType(),True)])
    artists = spark.createDataFrame(topArtists,IntegerType())
    final=artists.join(df_artist,artists.value==df_artist.artist_id).select(df_artist.artist_id,df_artist.artist_name)
    return final


# A better implementation using explode
def recommended_artist(als_model,user_id,limit):
    seletecd_user = model.recommendForAllUsers(limit).filter(col('user_id')==user_id)

    final = seletecd_user \
        .select("recommendations") \
        .withColumn('recommend', F.explode('recommendations')) \
        .select('recommend.*') \
        .join(df_artist, ['artist_id']) \
        .select('artist_id', 'artist_name')

    return final

In [30]:
#Write your code here
recommended_artist(model,2062243,20).show(truncate=False)

+---------+------------------------+
|artist_id|artist_name             |
+---------+------------------------+
|1001487  |Finch                   |
|1006633  |Coheed and Cambria      |
|1005386  |Stabbing Westward       |
|1011083  |Sonata Arctica          |
|407      |Eurythmics              |
|1000199  |Type O Negative         |
|4531     |"Weird Al" Yankovic     |
|1000052  |The Offspring           |
|2600     |Fear Factory            |
|3106     |Sean Paul               |
|1000183  |Disturbed               |
|5837     |Underworld              |
|1006672  |Further Seems Forever   |
|1270     |Queen                   |
|1001819  |2Pac                    |
|1191     |Elbow                   |
|5734     |Rob Zombie              |
|1001534  |Misfits                 |
|1066440  |P!nk                    |
|1004274  |The All-American Rejects|
+---------+------------------------+

